## Training Speech AI with Mozilla Data Collective

This tutorial walks you through downloading, loading and finetuning Whisper on an MDC dataset end to end. In this notebook we will use as an example use-case the [Khmer](https://en.wikipedia.org/wiki/Khmer_language) language (an official and national language in Cambodia)  using the [Khmer ASR Cultural Dataset](https://datacollective.mozillafoundation.org/datasets/cmkcy8in2004umo0775mye43g) steward by [Digital Divide Data](https://www.digitaldividedata.com/)

Specifically, the flow of this tutorial focuses on:

1. Ensuring a GPU set up is available
2. Setting up and logging in to Mozilla Data Collective
3. Downloading and loading the dataset as a pandas DataFrame - with a single function call!
4. Generating an automated Exploratory Data Analysis (EDA) report to understand the dataset and inform our finetuning configuration
5. Configuring Whisper's fine-tuning hyper-parameters
6. Launching a fine-tuning job!

Note that steps 1-4 are only required the first time you set up your environment and download the dataset. Once you have the dataset downloaded and ready to use, you can skip directly to steps 5: to create a new set of hyper-parameters and 6: to start a new finetuning job!


[![Try Finetuning on Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mozilla-Data-Collective/speech-to-text-finetune/blob/mdc-demo/demo/khmer.ipynb) Run this notebook in Google Collab for limited, free GPU access!


### 1. (Optional) Google Collab setup and GPU check

If you are running this notebook in Google Collab you'll need to enable GPUs for the notebook: Navigate to Edit→Notebook Settings Select T4 GPU from the Hardware Accelerator section Click Save and accept. Next, we'll confirm that we can connect to the GPU:

In [ ]:
import torch

if not torch.cuda.is_available():
    print("GPU NOT available!")
else:
    print("GPU is available!")

_**Do not run the following command if you are not in a Google Collab environment**_

Next we will need to install the required dependencies for this notebook for this runtime environment. Run the follow command:

In [ ]:
!git clone -b mdc-demo --quiet https://github.com/Mozilla-Data-Collective/speech-to-text-finetune.git
%cd speech-to-text-finetune

!python -m pip install -U pip uv
!uv pip install --system -e . --group demo

### 2. Setup and login to Mozilla Data Collective

***(Required)*** In order to download any MDC dataset you will need to first create an account at Mozilla Data Collective and then get an API key.

1. Create a Mozilla Data Collective [account](https://datacollective.mozillafoundation.org/)
2. Get your API key by following the instructions [here](https://datacollective.mozillafoundation.org/api-reference)
3. Set your API key as an environment variable in your .env file or export it in your terminal. If you are running this notebook in Google Collab, you can set it for the session by running the cell below and entering your API key when prompted.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # this will load environment variables from your .env file if you have one

if os.getenv(
    "MDC_API_KEY"
):  # if API key is not set you can set it through the interactive jupyter terminal
    print("MDC API key found and loaded successfully!")
else:
    import getpass

    os.environ["MDC_API_KEY"] = getpass.getpass("Enter your MDC API key: ")

***(Optional)*** If you want to track training and evaluation metrics of the finetuning and save your final model to use it and share it with others later, you will need a Hugging Face (HF) account.
1. Create a HF [account](https://huggingface.co/join)
2. Set up [personal access token](huggingface.co/settings/tokens)
3. Login to hugging face in this notebook by running the command below and using your token


In [ ]:
!huggingface-cli login

### 3. Download and load the Khmer dataset as a DataFrame

In [ ]:
from datacollective import load_dataset

dataframe = load_dataset(
    dataset_id="khmer-asr-cultural-dataset-4e33cd05",
    download_directory="local_data",
    enable_logging=True,
)

print("MDC dataframe preview:", dataframe.head())

### 4. Exploratory Data Analysis (EDA) with ydata-profiling

In [ ]:
from pathlib import Path

if Path("khmer_dataset_report.html").exists():
    print("EDA report already exists, skipping generation...")
else:
    from ydata_profiling import ProfileReport

    # The audio_path values are causing an issue with the library rendering the report so we remove them
    dataframe_no_audio_path = dataframe.drop(columns=["audio_path"])

    profile = ProfileReport(dataframe_no_audio_path, title="Profiling Report")
    profile.to_file("khmer_dataset_report.html")

    del dataframe_no_audio_path  # we delete the intermediate dataframe to save memory
del dataframe

### 5. Configure hyper-parameters for finetuning

In [ ]:
# @title Finetuning configuration and hyperparameter setting
import yaml


def save_to_yaml(filename="config.yaml"):
    with open(filename, "w") as file:
        yaml.dump(cfg, file)


model_id = "openai/whisper-tiny"  # @param ["openai/whisper-tiny", "openai/whisper-small", "openai/whisper-medium","openai/whisper-large-v3"]
dataset_id = "khmer-asr-cultural-dataset-4e33cd05"  # @param {type: "string"}
language = "Khmer"  # @param {type: "string"}
n_train_samples = 25  # @param {type: "int"}
n_test_samples = 10  # @param {type: "int"}
max_steps = 25  # @param {type: "slider", min: 1, max: 3000, step: 10}
per_device_train_batch_size = 8  # @param {type: "slider", min: 1, max: 300}
per_device_eval_batch_size = 4  # @param {type: "slider", min: 1, max: 200}
save_steps = 5  # @param {type: "slider", min: 1, max: 500}
logging_steps = 5  # @param {type: "slider", min: 1, max: 500}
load_best_model_at_end = True  # @param {type: 'boolean'}


gradient_checkpointing = True  # @param {type: 'boolean'}
fp16 = True  # @param {type: 'boolean'}
warmup_steps = 5  # @param {type: "slider", min: 0, max: 500}
gradient_accumulation_steps = 1  # @param {type: "slider", min: 1, max: 10}
download_directory = "local_data"  # @param {type: "string"}
# Optional HF repo name
repo_name = "default"  # @param {type: "string"}
push_to_hub = False  # @param {type: 'boolean'}
hub_private_repo = True  # @param {type: 'boolean'}

cfg = {
    "model_id": model_id,
    "dataset_id": dataset_id,
    "language": language,
    "repo_name": repo_name,
    "n_train_samples": n_train_samples,
    "n_test_samples": n_test_samples,
    "download_directory": download_directory,
    "training_hp": {
        "push_to_hub": push_to_hub,
        "hub_private_repo": hub_private_repo,
        "max_steps": max_steps,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": 1e-5,
        "warmup_steps": warmup_steps,
        "gradient_checkpointing": gradient_checkpointing,
        "fp16": fp16,
        "eval_strategy": "steps",
        "per_device_eval_batch_size": per_device_eval_batch_size,
        "predict_with_generate": True,
        "generation_max_length": 225,
        "save_steps": save_steps,
        "logging_steps": logging_steps,
        "load_best_model_at_end": load_best_model_at_end,
        "save_total_limit": 1,
        "metric_for_best_model": "wer",
        "greater_is_better": False,
    },
}

save_to_yaml()

### 6. Start finetuning job

Note that this might take a while, anything from 10min to 10hours depending on your model choice and hyper-parameter configuration

In [ ]:
from speech_to_text_finetune.finetune_whisper import run_finetuning

run_finetuning(config_path="config.yaml")